# Obstacles: how much they cover and what they cost

Every PASSAGE scene carries procedural obstacles: super-ellipses that block cells, cover up to 30 % of
the crop and never touch the start or the goal. This notebook shows what the masks look like, checks
how closely the drawn coverage follows its target, and measures their effect on the optimal routes over
the whole validation split of one configuration. The route effects come from stored costs, so no
tensor needs decoding.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads the metadata of one validation split (40,000 rows at 256×256) and six preview rows; about half a minute.

In [ ]:
resolution = 256
split = "validation"
gallery = 6  # preview scenes shown in Figure 1
output_dir = "notebook-artifacts/notebooks/obstacles"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2ab44d899652c7fb99c425488e84a7d565963547"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. What the masks look like

Figure 1 picks six preview scenes, from the least to the most covered, and draws their obstacles over
the terrain.

In [ ]:
previews = list(source.iter_rows(resolution, split, columns=["id", "tensor", "metadata"], samples=True, limit=64))
previews.sort(key=lambda r: r["metadata"]["obstacles"]["ratio_actual"])
picked = [previews[int(i)] for i in np.linspace(0, len(previews) - 1, gallery)]
fig, axes = plt.subplots(2, (gallery + 1) // 2, figsize=(11, 7.2))
fig.subplots_adjust(top=0.86, hspace=0.18, wspace=0.08)
for ax, preview in zip(axes.flat, picked):
    elevation, obstacles, start, goal = decode_scene(preview)
    render_scene(elevation, obstacles, start, goal, {}, ax=ax, legend=False, colorbar=False)
    ax.set_title(f"{preview['id']} · {100 * obstacles.mean():.1f} % blocked", fontsize=9)
figure_title(fig, "Obstacle masks from least to most covered",
             f"{gallery} of the 64 validation preview scenes · {resolution}×{resolution}")
figures = save_figure(fig, output_path, f"obstacles-{resolution}-gallery")
plt.show()

*Figure 1. Six validation scenes ordered by obstacle coverage. Hatched shapes are blocked cells; the
white circle and star are the start and the goal.*

## 2. Coverage against its target

The generator draws a target coverage for each scene, then places shapes until it gets close. The
census below reads the metadata of every scene of the split.

In [ ]:
SOLVERS = {"astar_elevation": "elevation", "astar_slope": "slope_uphill", "astar_energy": "energy"}
records = []
for row in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
    meta = row["metadata"]
    record = {"id": row["id"], "target": meta["obstacles"]["ratio_target"], "actual": meta["obstacles"]["ratio_actual"]}
    for solver, objective in SOLVERS.items():
        paths = meta["paths"][solver]
        record[f"{objective}_detour"] = (100 * (paths["obstacles"]["cost"] / paths["free"]["cost"] - 1)
                                         if paths["obstacles"]["status"] == "ok" else np.nan)
    records.append(record)
census = pd.DataFrame(records)

fig, (ax_hist, ax_target) = plt.subplots(1, 2, figsize=(11, 3.8))
fig.subplots_adjust(top=0.78, wspace=0.25)
ax_hist.hist(100 * census.actual, bins=np.linspace(0, 31, 32), color=REGIME_COLORS["obstacles"], alpha=0.8,
             edgecolor="white", linewidth=0.6)
ax_hist.set(xlabel="Obstacle coverage (% of cells)", ylabel="Scenes", title="Coverage")
ax_target.scatter(100 * census.target, 100 * census.actual, s=3, alpha=0.25, color=REGIME_COLORS["obstacles"],
                  linewidths=0, rasterized=True)
ax_target.plot([0, 31], [0, 31], color=MUTED, linewidth=1, linestyle=":")
ax_target.set(xlabel="Target coverage (%)", ylabel="Drawn coverage (%)", title="Drawn against target")
ax_target.grid(True, axis="both")
figure_title(fig, "How much the obstacles cover", f"{len(census):,} {split} scenes · {resolution}×{resolution}")
figures += save_figure(fig, output_path, f"obstacles-{resolution}-coverage")
plt.show()
gap = 100 * (census.actual - census.target)
display(Markdown(f"Drawn coverage stays within **{gap.abs().quantile(0.95):.2f} points** of its target for 95 % "
                 f"of the scenes; it is on average {abs(gap.mean()):.2f} points {'below' if gap.mean() < 0 else 'above'} it."))

*Figure 2. Left: obstacle coverage over the split. Right: drawn coverage against its target; the dotted
line marks equality.*

## 3. What obstacles do to the optimal route

The detour is the extra cost of the obstacle-aware optimum over the free-terrain optimum of the same
scene and objective.

In [ ]:
bins = pd.cut(100 * census.actual, [0, 5, 10, 15, 20, 25, 31], include_lowest=True,
              labels=["0–5", "5–10", "10–15", "15–20", "20–25", "25–31"])
rows = []
for objective in SOLVERS.values():
    for interval, group in census.groupby(bins, observed=True):
        rows.append({"Objective": OBJECTIVE_LABELS[objective], "Coverage (%)": str(interval),
                     "Scenes": len(group), "Median detour (%)": group[f"{objective}_detour"].median()})
effect = pd.DataFrame(rows)
show(effect, "Table 1. Detour by obstacle coverage.", "{:,.2f}")

fig, ax = plt.subplots(figsize=(8.6, 3.6))
fig.subplots_adjust(top=0.78)
for objective in SOLVERS.values():
    part = effect[effect.Objective == OBJECTIVE_LABELS[objective]]
    ax.plot(range(len(part)), part["Median detour (%)"], marker="o", markersize=5,
            color=OBJECTIVE_COLORS[objective], label=OBJECTIVE_LABELS[objective])
ax.set_xticks(range(len(part)), part["Coverage (%)"])
ax.set(xlabel="Obstacle coverage (% of cells)", ylabel="Median detour (%)")
ax.legend(fontsize=8)
figure_title(fig, "More obstacles, longer detours", f"{len(census):,} {split} scenes · {resolution}×{resolution}")
figures += save_figure(fig, output_path, f"obstacles-{resolution}-detour")
plt.show()

*Figure 3. Median detour against obstacle coverage, per objective. Each point groups the scenes of one
coverage band.*

## Summary

In [ ]:
changed ={o: census[f"{o}_detour"][census[f"{o}_detour"] > 1e-9] for o in SOLVERS.values()}
display(Markdown("\n".join([
    f"- Over {len(census):,} {split} scenes, obstacles cover {100 * census.actual.mean():.1f} % of the cells on "
    f"average, with a maximum of {100 * census.actual.max():.1f} %.",
    "- The obstacles raise the optimal cost in " + ", ".join(
        f"{100 * len(v) / len(census):.0f} % of the scenes ({OBJECTIVE_LABELS[o].lower()})" for o, v in changed.items())
    + "; in those scenes the median detour is " + ", ".join(
        f"{v.median():.2f} % ({OBJECTIVE_LABELS[o].lower()})" for o, v in changed.items()) + ".",
])))
report = {"resolution": resolution, "split": split, "scenes": len(census),
          "mean_coverage": float(census.actual.mean()), "effect_by_coverage": effect.to_dict("records"),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"obstacles-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*